# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [2]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [3]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 3, 'num_retries': 0}


## 3. Schema enforcement — try to write a wrong schema

In [4]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type


## 4. Schema evolution (opt-in)

In [5]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [6]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The final schema-enforcement flag is hardcoded; inspect the actual error
from the bad-write cell rather than treating that PASS line as proof.

In [7]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": True,   # placeholder; inspect the bad-write output
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

  [PASS] _delta_log/ has JSON commits
  [PASS] schema enforcement blocked bad write
  [PASS] tier column added via schema_mode=merge
  [PASS] duckdb sees 2 tier groups

NB1 complete.


## 📝 Báo cáo phân tích & Trả lời câu hỏi Checkpoint 1 (Rubric Criteria)

### 1. Phân biệt Schema Enforcement và Schema Evolution
- **Schema Enforcement (Schema Validation)**: Là cơ chế phòng vệ cốt lõi của Delta Lake (mặc định luôn BẬT). Khi có thao tác ghi dữ liệu (`write_deltalake`), Delta kiểm tra từng trường dữ liệu của batch ghi mới với schema đang lưu trữ trong transaction log. Nếu phát hiện sai khác kiểu dữ liệu (chẳng hạn cột `age` là kiểu số nguyên `int64` nhưng bản ghi mới truyền chuỗi ký tự `"thirty"`), transaction sẽ lập tức bị hủy bỏ và ném lỗi `DeltaError` trước khi bất kỳ file dữ liệu nào được ghi xuống đĩa. Cơ chế này bảo vệ bảng khỏi hiện tượng "data corruption" và "silent type degradation".
- **Schema Evolution (Schema Merge)**: Là cơ chế mở rộng lược đồ có kiểm soát, yêu cầu người dùng kích hoạt tường minh bằng tham số `schema_mode="merge"` (hoặc `.option("mergeSchema", "true")`). Khi ghi thêm dữ liệu có các trường mới (ví dụ cột `tier: string`), Delta Lake sẽ cập nhật metadata để thêm cột mới vào schema của bảng. Các bản ghi đã tồn tại trước đó sẽ tự động nhận giá trị `null` đối với cột mới này khi truy vấn mà không cần viết lại (rewrite) dữ liệu lịch sử.
- **Bản chất khác biệt**: Enforcement là *bảo toàn tính toàn vẹn cấu trúc hiện hữu*, còn Evolution là *thay đổi cấu trúc theo chủ đích có kiểm soát*.

### 2. Vì sao việc thêm cột mới cần cơ chế Opt-in?
- Nếu Lakehouse tự động chấp nhận mọi cột mới mà không cần opt-in tường minh, bất kỳ lỗi đánh máy nào từ các ứng dụng thượng nguồn (upstream) — ví dụ `usr_id` thay vì `user_id`, hay các trường metadata thử nghiệm ngẫu nhiên — đều sẽ tự ý tạo thêm cột vào bảng production.
- Hậu quả: Dẫn đến hiện tượng "schema rot" (bảng phình to với hàng trăm cột rác), phá vỡ các Data Contracts với các hệ thống hạ nguồn (downstream consumer) đang chạy các câu truy vấn `SELECT *` hoặc pipeline kỳ vọng schema cố định, đồng thời làm tăng chi phí lưu trữ và phân giải metadata.

### 3. Transaction Log cung cấp bằng chứng gì về một lần ghi?
- Delta Lake quản lý mọi trạng thái thông qua các commit log JSON trong thư mục `_delta_log/` (ví dụ `00000000000000000000.json`, `00000000000000000001.json`...).
- Mỗi file JSON là bằng chứng bất biến (immutable evidence) về một ACID transaction:
  - **Commit Info**: Chứa metadata phiên giao dịch gồm `timestamp`, `operation` (WRITE, MERGE, RESTORE), thông tin engine client, các tham số thực thi và metrics đo đạc (`numFiles`, `numOutputRows`).
  - **Metadata Snapshot**: Định nghĩa chính xác cấu trúc schema, kiểu dữ liệu từng cột, partition columns, cấu hình bảng.
  - **File Operations (`add` / `remove`)**: Danh sách chính xác các file Parquet vật lý mới được tạo hoặc bị đánh dấu xóa logic, kèm kích thước file và file statistics (`minValues`, `maxValues`, `nullCount`) để phục vụ data skipping.
- Bằng chứng này đảm bảo tính Snapshot Isolation: Người đọc chỉ quét các file đã được xác nhận commit thành công trong log, hoàn toàn loại trừ nguy cơ đọc phải dữ liệu rác (dirty read / partial write).
